**Data Integrity check**

- Verify that all image files exist and match the IDs in the metadata CSV file
- Detect missing images, incorrect filenames, or corrupted dataset structure before training

In [1]:
import os
import pandas as pd

In [2]:
train_path = "../data/raw/train-metadata.csv"
test_path = "../data/raw/test-metadata.csv"

train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path)

# Basic Info

### Dataset Size

- Train set contains a large number of samples (~400k)
- Test set is extremely small (only a few samples), suggesting it may be a placeholder or subset for demonstration purposes

In [3]:
print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

Train samples: 401059
Test samples: 3


## Preview data

### Note: 
* Some features are only available in the training set. 

* Only common features between train and test are used for model training and inference.

In [4]:
train_df.head()

,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,0,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,0,IP_4111386,65.0,male,anterior torso,3.22,TBP tile: close-up,3D: XP,14.242329,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,0,IP_8313778,55.0,male,anterior torso,2.73,TBP tile: close-up,3D: white,24.725520,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510


In [5]:
train_df.tail()

,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
401054,ISIC_9999937,0,IP_1140263,70.0,male,anterior torso,6.80,TBP tile: close-up,3D: XP,22.574335,...,IL_9520694,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999988
401055,ISIC_9999951,0,IP_5678181,60.0,male,posterior torso,3.11,TBP tile: close-up,3D: white,19.977640,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999820
401056,ISIC_9999960,0,IP_0076153,65.0,female,anterior torso,2.05,TBP tile: close-up,3D: XP,17.332567,...,IL_9852274,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999416
401057,ISIC_9999964,0,IP_5231513,30.0,female,anterior torso,2.80,TBP tile: close-up,3D: XP,22.288570,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,100.000000
401058,ISIC_9999967,0,IP_6426047,50.0,male,lower extremity,3.30,TBP tile: close-up,3D: XP,16.792900,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999960


In [6]:
test_df.head()

,isic_id,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,...,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license
0,ISIC_0015657,IP_6074337,45.0,male,posterior torso,2.70,TBP tile: close-up,3D: XP,22.80433,20.007270,...,0.304827,1.281532,2.299935,0.479339,20,-155.06510,1511.222000,113.980100,Memorial Sloan Kettering Cancer Center,CC-BY
1,ISIC_0015729,IP_1664139,35.0,female,lower extremity,2.52,TBP tile: close-up,3D: XP,16.64867,9.657964,...,0.000000,1.271940,2.011223,0.426230,25,-112.36924,629.535889,-15.019287,"Frazer Institute, The University of Queensland...",CC-BY
2,ISIC_0015740,IP_7142616,65.0,male,posterior torso,3.16,TBP tile: close-up,3D: XP,24.25384,19.937380,...,0.230742,1.080308,2.705857,0.366071,110,-84.29282,1303.978000,-28.576050,FNQH Cairns,CC-BY


### Missing Values Analysis

Several features contain a high proportion of missing values:

- Nearly entirely missing:
  - `iddx_*`
  - `mel_mitotic_index`
  - `mel_thick_mm`

- Moderately missing:
  - `lesion_id`

- Slightly missing:
  - `sex`, `age_approx`, `anatom_site_general`

### Decision

- Features with extremely high missing rates will likely be **dropped**
- Features with moderate/low missing values may be **imputed**

In [7]:
print("\n===== MISSING VALUES (TRAIN) =====")
print(train_df.isnull().sum().sort_values(ascending=False).head(10))

print("\n===== MISSING VALUES (TEST) =====")
print(test_df.isnull().sum().sort_values(ascending=False).head(10))


===== MISSING VALUES (TRAIN) =====
iddx_5                 401058
mel_mitotic_index      401006
mel_thick_mm           400996
iddx_4                 400508
iddx_3                 399994
iddx_2                 399991
lesion_id              379001
sex                     11517
anatom_site_general      5756
age_approx               2798
dtype: int64

===== MISSING VALUES (TEST) =====
isic_id                   0
patient_id                0
age_approx                0
sex                       0
anatom_site_general       0
clin_size_long_diam_mm    0
image_type                0
tbp_tile_type             0
tbp_lv_A                  0
tbp_lv_Aext               0
dtype: int64


In [8]:
print("\n===== DUPLICATES =====")
print("Train duplicates:", train_df.duplicated().sum())
print("Test duplicates:", test_df.duplicated().sum())


===== DUPLICATES =====
Train duplicates: 0
Test duplicates: 0


### Label Distribution (Critical)

The dataset is **extremely imbalanced**:

- Benign (0): ~99.9%
- Malignant (1): ~0.1%

### Implications

- Accuracy is NOT a reliable metric
- Model may bias heavily toward the majority class

### Strategy

- Use **stratified splitting**
- Apply **class weighting or focal loss**
- Evaluate using:
  - ROC-AUC
  - PR-AUC

In [9]:
print("\n===== LABEL DISTRIBUTION =====")
print(train_df["target"].value_counts(normalize=True))


===== LABEL DISTRIBUTION =====
target
0    0.99902
1    0.00098
Name: proportion, dtype: float64


### Train vs Test Feature Mismatch

Some features exist only in the training set:

- `iddx_*`
- `mel_*`
- `tbp_lv_dnn_lesion_confidence`
- `target` (expected)

### Risk

Using these features may cause **data leakage**, since they are not available during inference.

### Decision

- Only use features that exist in BOTH train and test sets

In [10]:
print("\n===== FEATURE CHECK =====")
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

print("Only in train:", train_cols - test_cols)
print("Only in test:", test_cols - train_cols)


===== FEATURE CHECK =====
Only in train: {'iddx_1', 'mel_mitotic_index', 'lesion_id', 'iddx_full', 'tbp_lv_dnn_lesion_confidence', 'iddx_3', 'iddx_2', 'mel_thick_mm', 'target', 'iddx_4', 'iddx_5'}
Only in test: set()


### Image Integrity Check

- No missing images detected
- All metadata entries correctly map to image files

This ensures the dataset is consistent and ready for training

In [11]:
image_folder = "../data/raw/ISIC_2024_Training_Input"

missing_images = []

for img_id in train_df["isic_id"]:
    img_path = os.path.join(image_folder, f"{img_id}.jpg")
    if not os.path.exists(img_path):
        missing_images.append(img_id)

print("\n===== IMAGE CHECK =====")
print("Missing images:", len(missing_images))


===== IMAGE CHECK =====
Missing images: 0


## Summary of Findings

- Dataset is **clean** (no duplicates, no missing images)
- Severe **class imbalance** (~0.1% positive class)
- Several features contain **extreme missing values**
- Some features exist only in train so they must be removed

## Next Steps

- Feature selection (align train/test)
- Handle missing values
- Encode categorical features
- Address class imbalance
- Build training pipeline